### Install pre-requistes


In [ ]:
!pip install --upgrade pip SPARQLWrapper pandas matplotlib ftfy --quiet 
import sys
!{sys.executable} -m pip uninstall -y py4cytoscape
!{sys.executable} -m pip install py4cytoscape

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

ENDPOINT_URL = "http://localhost:7200/repositories/KG-Questionnaries-Core"

PREFIXES = """\
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX ds:   <http://digistrucmed.org/questionnaire#>
"""

In [ ]:
def run_query(query: str) -> pd.DataFrame:
    """Run a SPARQL SELECT query and return the results as a Pandas DataFrame."""
    sparql = SPARQLWrapper(ENDPOINT_URL)
    sparql.setReturnFormat(JSON)
    sparql.setQuery(PREFIXES + "\n" + query)
    results = sparql.query().convert()

    cols = results["head"].get("vars", [])
    rows = []
    for b in results["results"]["bindings"]:
        row = {}
        for c in cols:
            val = b.get(c, {}).get("value")
            row[c] = val
        rows.append(row)
    return pd.DataFrame(rows, columns=cols)

def short_uri(uri):
    """Return the local name of a URI (the part after the last '#' or '/')."""
    if uri is None or (isinstance(uri, float) and pd.isna(uri)):
        return uri
    uri = str(uri)
    return uri.rsplit("#", 1)[-1] if "#" in uri else uri.rsplit("/", 1)[-1]

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_colwidth", 120)


## 1. Schema Overview
What is the graph

In [ ]:
q1 = """\
SELECT ?type (COUNT(?s) AS ?count)
WHERE { ?s a ?type . }
GROUP BY ?type
ORDER BY DESC(?count)
"""
df1 = run_query(q1)
df1["count"] = pd.to_numeric(df1["count"], errors="coerce")
df1[["type", "count"]]


## 2. Predicate Usage

How nodes are connected.


In [ ]:
q2 = """\
SELECT ?predicate (COUNT(*) AS ?count)
WHERE { ?s ?predicate ?o .
FILTER(STRSTARTS(STR(?predicate), STR(ds:)))}
GROUP BY ?predicate
ORDER BY DESC(?count)
"""

df2 = run_query(q2)
df2["count"] = pd.to_numeric(df2["count"], errors="coerce")
df2["percentage"] = (df2["count"] / df2["count"].sum() * 100).round(2)
df2[["predicate", "count", "percentage"]]


In [ ]:
if not df2.empty:
    df2_plot = df2.dropna(subset=["count"]).sort_values("count", ascending=False)
    plt.figure(figsize=(10, 4))
    plt.xticks(rotation=45, ha="right")
    plt.title("Instances per Questionnaire class")
    plt.ylabel("Count")
    plt.xlabel("Class")
    bars = plt.bar(df2_plot["predicate"], df2_plot["count"])
    plt.bar_label(bars, padding=3)
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot for query 2.")


## 3. Walk from a Question to its answers

Explore the number of each answer to each question.


In [ ]:
q3 = """\
SELECT ?questionDesc ?category (COUNT(DISTINCT ?resp) AS ?numPeople)
WHERE {
  ?question a ds:Question ; ds:QuestionDescription ?questionDesc .
  ?resp a ds:Answered ; ds:hasAnswer ?answer .
  ?answer ds:isAnswerTo ?question ; ds:hasAnswerCategory ?category .
}
GROUP BY ?questionDesc ?category
ORDER BY DESC (?numPeople)
"""
df3 = run_query(q3)
df3["numPeople"] = pd.to_numeric(df3["numPeople"], errors="coerce")

# Keep the raw query result visible first.
display(df3.head(20))

# Compact KG-oriented summary: how many category entries and respondents each question has.
summary3 = (
    df3.groupby("questionDesc", as_index=False)
    .agg(
        categories=("category", "nunique"),
        total_respondents=("numPeople", "sum"),
        avg_respondents=("numPeople", "mean"),
        max_category_count=("numPeople", "max"),
    )
    .sort_values("total_respondents", ascending=False)
    .round(2)
    .reset_index(drop=True)
)

display(summary3.head(20))



In [ ]:
import html
import re
import textwrap

def clean_label(value):
    """Normalize question text for display in tables and plots."""
    if pd.isna(value):
        return value

    text = html.unescape(str(value)).strip()
    text = " ".join(text.split())

    # First try to repair mojibake with a deterministic decode path.
    candidates = [text]
    for encoding in ("latin1", "cp1252"):
        try:
            candidates.append(text.encode(encoding).decode("utf-8"))
        except UnicodeError:
            pass

    try:
        from ftfy import fix_text
        candidates.append(fix_text(text))
    except ImportError:
        pass

    # Apply a small fallback dictionary for labels that were already replaced
    # with the Unicode replacement character in the source export.
    replacements = {
        "Verst�ndnis": "Verständnis",
        "w�rden": "würden",
        "Verf�gung": "Verfügung",
        "h�ufiger": "häufiger",
        "f�hlte": "fühlte",
        "k�dnte": "könnte",
        "n�tigen": "nötigen",
        "n�tig": "nötig",
        "umst�ndlich": "umständlich",
        "unterst�tzung": "unterstützung",
    }

    def apply_replacements(candidate):
        for broken, fixed in replacements.items():
            candidate = candidate.replace(broken, fixed)
        return candidate

    candidates = [apply_replacements(candidate) for candidate in candidates]
    candidates = [candidate for candidate in candidates if candidate]

    def score(candidate):
        penalty = candidate.count("�") * 10
        penalty += len(re.findall(r"[ÃÂ]", candidate)) * 5
        penalty += sum(candidate.count(token) for token in ("Verst�ndnis", "w�rden", "Verf�gung", "h�ufiger", "f�hlte", "k�dnte", "umst�ndlich")) * 20
        return penalty

    text = min(candidates, key=score) if candidates else text
    return text

summary3["question_label"] = summary3["questionDesc"].map(clean_label)
summary3["top_category_count"] = summary3["max_category_count"]

display(
    summary3[["question_label", "categories", "total_respondents", "avg_respondents", "top_category_count"]]
    .head(20)
    .rename(columns={
        "question_label": "question",
        "categories": "num_categories",
        "total_respondents": "total_distinct_respondents",
        "avg_respondents": "avg_respondents_per_category",
        "top_category_count": "largest_category_count",
    })
)

# Use the top 10 questions, but show the full category distribution for each one.
top_questions = summary3.head(10).sort_values("total_respondents", ascending=True).copy()
top_questions["plot_label"] = top_questions["question_label"].map(lambda value: textwrap.fill(value, width=58))

category_order = [
    "http://digistrucmed.org/questionnaire#NoInfo",
    "http://digistrucmed.org/questionnaire#very_negative",
    "http://digistrucmed.org/questionnaire#negative",
    "http://digistrucmed.org/questionnaire#neutral",
    "http://digistrucmed.org/questionnaire#positive",
    "http://digistrucmed.org/questionnaire#very_positive",
    "http://digistrucmed.org/questionnaire#Format",
]
available_categories = [category for category in category_order if category in df3["category"].unique()]

top_question_matrix = (
    df3[df3["questionDesc"].isin(top_questions["questionDesc"])]
    .pivot_table(
        index="questionDesc",
        columns="category",
        values="numPeople",
        fill_value=0,
        aggfunc="sum",
    )
    .reindex(index=top_questions["questionDesc"], columns=available_categories, fill_value=0)
    .rename(index=dict(zip(top_questions["questionDesc"], top_questions["plot_label"])))
)

fig, ax = plt.subplots(figsize=(14, 8), dpi=150)
left = pd.Series(0, index=top_question_matrix.index)

palette = {
    "http://digistrucmed.org/questionnaire#NoInfo": "#9e9e9e",
    "http://digistrucmed.org/questionnaire#very_negative": "#b2182b",
    "http://digistrucmed.org/questionnaire#negative": "#ef8a62",
    "http://digistrucmed.org/questionnaire#neutral": "#fddbc7",
    "http://digistrucmed.org/questionnaire#positive": "#80cdc1",
    "http://digistrucmed.org/questionnaire#very_positive": "#1a9850",
    "http://digistrucmed.org/questionnaire#Format": "#5b5b5b",
}

for category in available_categories:
    values = top_question_matrix[category]
    ax.barh(
        top_question_matrix.index,
        values,
        left=left,
        label=short_uri(category),
        color=palette.get(category, "#2b7bbb"),
        edgecolor="none",
)
    left = left + values

ax.set_title("Top 10 questions by distinct respondents and category", loc="left", pad=12, fontsize=15, fontweight="bold")
ax.set_xlabel("Distinct respondents")
ax.set_ylabel("Question")
ax.grid(axis="x", linestyle="--", alpha=0.25)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
ax.margins(x=0.05)

plt.tight_layout()
plt.show()

# Heatmap-style table: useful to inspect question/category coverage in the KG.
display(top_question_matrix)

# Compact category totals across the entire KG.
category_summary = (
    df3.groupby("category", as_index=False)
    .agg(
        questions=("questionDesc", "nunique"),
        total_distinct_respondents=("numPeople", "sum"),
        avg_distinct_respondents=("numPeople", "mean"),
    )
    .sort_values("total_distinct_respondents", ascending=False)
    .round(2)
    .reset_index(drop=True)
)

display(category_summary)

## 4. Trace one participant's full path through the graph
Trace the answers submitted by the participant with UserId=100.

In [ ]:
q4 = """
SELECT ?questionDesc ?rawResponseValue ?category ?questionnaire
WHERE {
  VALUES ?user { ds:100 }   # swap in any UserId
  ?resp a ds:Answered ; ds:answeredByUser ?user ;
        ds:rawResponseValue ?rawResponseValue ;
        ds:answeredInQuestionnaire ?questionnaire ;
        ds:hasAnswer ?answer .
    ?answer ds:isAnswerTo ?question ; ds:hasAnswerCategory ?category .
  ?question ds:QuestionDescription ?questionDesc .
}
ORDER BY ?questionnaire ?questionDesc
"""
df4 = run_query(q4)
df4["questionDesc"] = df4["questionDesc"].map(clean_label)
df4[["questionDesc", "rawResponseValue", "category", "questionnaire"]]

## 4.Find gaps (orphan/disconnected nodes)

In [ ]:
q4 = """
SELECT ?question ?questionDesc
WHERE {
  ?question a ds:Question ; ds:QuestionDescription ?questionDesc .
  FILTER NOT EXISTS { ?resp a ds:Answered ; ds:hasAnswer/ds:isAnswerTo ?question . }
}
"""
df4 = run_query(q4)
df4["questionDesc"] = df4["questionDesc"].map(clean_label)
df4[["question", "questionDesc"]]

## 4.Network Degree Centrality of Questionnaire Entities
Using predicate edges.

In [ ]:
# Build a Questionnaire relations graph
q_edges = """
SELECT DISTINCT ?s ?p ?o
WHERE {
  ?s ?p ?o .
  FILTER(isIRI(?s) && isIRI(?o))
  FILTER(STRSTARTS(STR(?p), "http://digistrucmed.org/questionnaire#"))
}
"""

q_node_types = """
SELECT DISTINCT ?node ?type
WHERE {
  ?node rdf:type ?type .
  FILTER(STRSTARTS(STR(?type), "http://digistrucmed.org/questionnaire#"))
}
"""

# Run queries
df_edges = run_query(q_edges)
df_node_types = run_query(q_node_types)

import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

def short_uri(x):
    x = str(x)
    return x.rsplit("#", 1)[-1] if "#" in x else x.rsplit("/", 1)[-1]

G = nx.Graph()
for _, r in df_edges.iterrows():
    G.add_edge(r["s"], r["o"], predicate=short_uri(r["p"]))

print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")

# Degree centrality
deg_centrality = nx.degree_centrality(G)

type_labels = df_node_types.copy()
type_labels["type_label"] = type_labels["type"].map(short_uri)

node_type_map = (
    type_labels.groupby("node")["type_label"]
    .apply(lambda values: ", ".join(sorted(set(values))))
    .to_dict()
)

df_deg = (
    pd.DataFrame(deg_centrality.items(), columns=["node", "centrality"])
      .assign(
          label=lambda d: d["node"].map(short_uri),
          type=lambda d: d["node"].map(node_type_map).fillna("Unknown")
      )
      .sort_values("centrality", ascending=False)
)

display(df_deg[["label", "type", "centrality", "node"]].head(20))

# Average number of neighbors
degrees = dict(G.degree())
avg_neighbors = sum(degrees.values()) / len(degrees) if degrees else 0
print("Average number of neighbors (overall):", avg_neighbors)

# Average neighbors by real questionnaire class
degree_df = pd.DataFrame(degrees.items(), columns=["node", "degree"])

typed_degree_df = type_labels.merge(degree_df, on="node", how="inner")

avg_neighbors_by_type = (
    typed_degree_df.groupby("type_label", as_index=False)
    .agg(
        nodes=("node", "nunique"),
        avg_neighbors=("degree", "mean"),
        max_neighbors=("degree", "max")
    )
    .sort_values("avg_neighbors", ascending=False)
)

display(avg_neighbors_by_type)

# Additional global metrics
if G.number_of_nodes() > 0:
    try:
        if nx.is_connected(G):
            asp = nx.average_shortest_path_length(G)
        else:
            largest_cc = max(nx.connected_components(G), key=len)
            H = G.subgraph(largest_cc).copy()
            asp = nx.average_shortest_path_length(H)

        print("Average shortest path length (largest component):", asp)
    except Exception as e:
        print("Could not compute average shortest path length:", e)

# Network Visualization
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=0)

nx.draw(G, pos, with_labels=False, node_size=50)
plt.title("Questionnaire-Node Graph")
plt.show()

## 5. Centrality analysis in Cytoscape (py4cytoscape)

This sends the questionnaire relation graph to a running Cytoscape desktop session, runs Cytoscape's NetworkAnalyzer, and retrieves the node centrality measures. Start Cytoscape before running this cell.

In [ ]:
import py4cytoscape as p4c

# Cytoscape must be open locally (Apps -> CyREST is included by default).
try:
    p4c.cytoscape_ping()
except Exception as exc:
    raise ConnectionError(
        'Cytoscape is not reachable. Open Cytoscape, wait for it to finish starting, then rerun this cell.'
    ) from exc

# Reuse df_edges created in Section 4. Cytoscape expects id/source/target columns.
nodes_cy = pd.DataFrame({'id': sorted(G.nodes())})
nodes_cy['label'] = nodes_cy['id'].map(lambda uri: str(uri).rsplit('#', 1)[-1])
edges_cy = df_edges[['s', 'o', 'p']].rename(
    columns={'s': 'source', 'o': 'target', 'p': 'predicate'}
)
edges_cy['interaction'] = edges_cy['predicate'].map(
    lambda uri: str(uri).rsplit('#', 1)[-1]
)

network_suid = p4c.create_network_from_data_frames(
    nodes=nodes_cy, edges=edges_cy,
    title='Questionnaire relation graph - centrality',
    collection='Questionnaire analysis'
)

# The graph was built as undirected in NetworkX, so analyse it as undirected here too.
p4c.set_current_network(network_suid)
p4c.analyze_network(directed=False)

# Retrieve the centrality results written by Cytoscape NetworkAnalyzer.
node_table = p4c.get_table_columns(table='node', network=network_suid)
centrality_metrics = [
    metric for metric in [
        'Degree', 'BetweennessCentrality', 'ClosenessCentrality',
        'Radiality', 'Stress', 'ClusteringCoefficient'
    ] if metric in node_table.columns
]

centrality_df = (
    node_table[['name'] + centrality_metrics]
    .rename(columns={'name': 'node'})
    .sort_values(
        by='BetweennessCentrality' if 'BetweennessCentrality' in centrality_metrics else 'Degree',
        ascending=False
    )
    .reset_index(drop=True)
)

display(centrality_df.head(20))
centrality_df.to_csv('questionnaire_cytoscape_centrality.csv', index=False)
print(f'Created Cytoscape network {network_suid} and saved centrality results for {len(centrality_df)} nodes.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def short_uri(x):
    if pd.isna(x):
        return x
    x = str(x)
    return x.split("#")[-1] if "#" in x else x.rsplit("/", 1)[-1]

# clean_label() is defined in Section 3 (repairs German mojibake).
# Fall back to identity if this cell is run in isolation.
_clean = clean_label if "clean_label" in globals() else (lambda v: v)

# Ordinal scale for the sentiment categories (1 = very negative .. 5 = very positive).
# NoInfo / Format are treated as non-scored (missing) for the numeric statistics.
CATEGORY_SCORE = {
    "very_negative": 1, "negative": 2, "neutral": 3, "positive": 4, "very_positive": 5,
}
CATEGORY_ORDER = ["very_negative", "negative", "neutral", "positive", "very_positive"]
POLARITY = {
    "very_negative": "negative", "negative": "negative",
    "neutral": "neutral",
    "positive": "positive", "very_positive": "positive",
}
POLARITY_COLOR = {"negative": "#d6604d", "neutral": "#f0e0a8", "positive": "#4393c3"}

def polarity_color(cat):
    return POLARITY_COLOR.get(POLARITY.get(cat), "#9e9e9e")

# ------------------------------------------------------------
# 1. Answer frequencies per question (now including the sentiment category)
# ------------------------------------------------------------
q_answer_stats = """
SELECT
    ?question
    ?questionText
    ?responseCode
    ?responseLabel
    ?category
    (COUNT(DISTINCT ?answered) AS ?frequency)
WHERE {
    ?answered ds:hasQuestion ?question ;
              ds:hasAnswer ?answer .

    OPTIONAL { ?question ds:QuestionDescription ?questionText . }
    OPTIONAL { ?answer ds:ResponseCode        ?responseCode . }
    OPTIONAL { ?answer ds:ResponseLabel       ?responseLabel . }
    OPTIONAL { ?answer ds:hasAnswerCategory   ?category . }
}
GROUP BY ?question ?questionText ?responseCode ?responseLabel ?category
ORDER BY ?question DESC(?frequency)
"""

df = run_query(q_answer_stats)

df["frequency"]      = pd.to_numeric(df["frequency"], errors="coerce").fillna(0).astype(int)
df["question_short"] = df["question"].apply(short_uri)
df["question_text"]  = df["questionText"].map(_clean)
df["category_short"] = df["category"].apply(short_uri)
df["answer"]         = df["responseLabel"].fillna(df["responseCode"]).fillna(df["category_short"])
df["polarity"]       = df["category_short"].map(POLARITY)
df["score"]          = df["category_short"].map(CATEGORY_SCORE)

df["n_question"]  = df.groupby("question_short")["frequency"].transform("sum")
df["percentage"]  = (df["frequency"] / df["n_question"] * 100).round(2)
df = df.sort_values(["question_short", "frequency"], ascending=[True, False])

print(f"{df['question_short'].nunique()} questions, "
      f"{int(df['frequency'].sum())} answers total")

display(
    df[["question_short", "question_text", "responseCode", "answer",
        "category_short", "frequency", "percentage"]].head(100)
)

# ------------------------------------------------------------
# 2. Per-question statistical summary
#    n, modal answer, mean/median/std sentiment, polarity shares,
#    and a response-diversity index (normalised entropy: 0 = consensus, 1 = uniform)
# ------------------------------------------------------------
def weighted_median(scores, freqs):
    order = np.argsort(scores)
    s = np.asarray(scores, float)[order]
    f = np.asarray(freqs, float)[order]
    cum = np.cumsum(f)
    idx = min(int(np.searchsorted(cum, f.sum() / 2.0)), len(s) - 1)
    return float(s[idx])

def norm_entropy(freqs):
    p = np.asarray(freqs, float)
    p = p[p > 0]
    if p.size <= 1:
        return 0.0
    p = p / p.sum()
    return float(-(p * np.log2(p)).sum() / np.log2(p.size))

def summarise(group):
    total = int(group["frequency"].sum())
    top = group.loc[group["frequency"].idxmax()]
    out = {
        "question_text": group["question_text"].iloc[0],
        "n_respondents": total,
        "n_options": int((group["frequency"] > 0).sum()),
        "modal_answer": top["answer"],
        "modal_pct": round(top["frequency"] / total * 100, 1) if total else np.nan,
    }

    scored = group.dropna(subset=["score"])
    if scored["frequency"].sum() > 0:
        s, f = scored["score"].values, scored["frequency"].values
        mean = np.average(s, weights=f)
        out["mean_score"]   = round(float(mean), 2)
        out["median_score"] = weighted_median(s, f)
        out["std_score"]    = round(float(np.sqrt(np.average((s - mean) ** 2, weights=f))), 2)
    else:
        out["mean_score"] = out["median_score"] = out["std_score"] = np.nan

    pol = group.groupby("polarity")["frequency"].sum()
    scored_total = pol.reindex(["negative", "neutral", "positive"]).fillna(0).sum()
    if scored_total:
        out["pct_negative"] = round(pol.get("negative", 0) / scored_total * 100, 1)
        out["pct_neutral"]  = round(pol.get("neutral", 0)  / scored_total * 100, 1)
        out["pct_positive"] = round(pol.get("positive", 0) / scored_total * 100, 1)
    else:
        out["pct_negative"] = out["pct_neutral"] = out["pct_positive"] = np.nan

    out["response_diversity"] = round(norm_entropy(group["frequency"].values), 2)
    return pd.Series(out)

summary = df.groupby("question_short").apply(summarise).reset_index()

print("Per-question summary (sorted by mean sentiment, most positive first):")
display(
    summary.sort_values("mean_score", ascending=False, na_position="last")
           .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Distribution for one question, ordered along the sentiment scale
# ------------------------------------------------------------
question_id = "M006_01"   # change this
one = df[df["question_short"] == question_id].copy()
one["order"] = one["category_short"].map({c: i for i, c in enumerate(CATEGORY_ORDER)})
one = one.sort_values(["order", "frequency"], ascending=[True, False])

display(one[["responseCode", "answer", "category_short", "frequency", "percentage"]])

if not one.empty:
    y = np.arange(len(one))
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(one) + 2))
    bars = ax.barh(y, one["frequency"],
                   color=[polarity_color(c) for c in one["category_short"]])
    ax.set_yticks(y)
    ax.set_yticklabels(one["answer"].astype(str))
    ax.invert_yaxis()
    for bar, pct in zip(bars, one["percentage"]):
        ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
                f" {pct:.1f}%", va="center", fontsize=9)
    q_txt = str(one["question_text"].iloc[0])
    ax.set_xlabel("Number of respondents")
    ax.set_ylabel("Answer")
    ax.set_title(f"{question_id} — {q_txt[:70]}", loc="left", fontsize=11)
    ax.margins(x=0.15)
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 4. Which questions scored most / least positively (mean sentiment vs. neutral)
# ------------------------------------------------------------
ranked = summary.dropna(subset=["mean_score"]).copy()
if not ranked.empty:
    ranked["delta"] = ranked["mean_score"] - 3.0          # distance from neutral (3)
    ranked = ranked.sort_values("delta")
    show = pd.concat([ranked.head(10), ranked.tail(10)]).drop_duplicates("question_short")
    show = show.sort_values("delta")

    y = np.arange(len(show))
    fig, ax = plt.subplots(figsize=(10, 0.42 * len(show) + 1.5))
    ax.barh(y, show["delta"],
            color=np.where(show["delta"] >= 0, POLARITY_COLOR["positive"], POLARITY_COLOR["negative"]))
    ax.axvline(0, color="0.4", lw=1)
    ax.set_yticks(y)
    ax.set_yticklabels(show["question_short"])
    for yi, (d, m) in enumerate(zip(show["delta"], show["mean_score"])):
        ax.text(d + (0.03 if d >= 0 else -0.03), yi, f"{m:.2f}",
                va="center", ha="left" if d >= 0 else "right", fontsize=8)
    ax.set_xlabel("Mean sentiment relative to neutral  (mean score − 3)")
    ax.set_title("Most positively / negatively rated questions", loc="left", fontsize=12)
    ax.margins(y=0.01)
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 5. Overall answer-category distribution across the whole questionnaire
# ------------------------------------------------------------
cat_totals = df.groupby("category_short")["frequency"].sum()
ordered_cats = ([c for c in CATEGORY_ORDER if c in cat_totals.index] +
                [c for c in cat_totals.index if c not in CATEGORY_ORDER])
overall = (cat_totals.reindex(ordered_cats)
                     .rename_axis("category").reset_index(name="frequency"))
overall["percentage"] = (overall["frequency"] / overall["frequency"].sum() * 100).round(2)
display(overall)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(overall["category"], overall["frequency"],
              color=[polarity_color(c) for c in overall["category"]])
for b, p in zip(bars, overall["percentage"]):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(),
            f"{p:.1f}%", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Responses")
ax.set_title("Overall answer-category distribution", loc="left", fontsize=12)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
